In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent

# путь до src
src_path = project_root / "src"

# добавляем src в пути импорта
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("PROJECT ROOT:", project_root)
print("SRC PATH:", src_path)

PROJECT ROOT: C:\Users\denis\credit_scoring
SRC PATH: C:\Users\denis\credit_scoring\src


In [39]:
from config.db_config import DB_ARGS, DATA_FULL_PATH
from app.utils.db_manager import PostgresDB

db = PostgresDB(DB_ARGS)
db.connect()

Подключение к БД успешно установлено


In [3]:
import os
from datetime import datetime

import pandas as pd
import psycopg2

In [4]:
from warnings import filterwarnings

filterwarnings('ignore', category=UserWarning, message='.*pandas only supports SQLAlchemy connectable.*')

In [29]:
sql_schema_query = """
DROP TABLE IF EXISTS bureau;

CREATE TABLE bureau(
    SK_ID_CURR integer,
    SK_ID_BUREAU integer,
    CREDIT_ACTIVE varchar(13),
    CREDIT_CURRENCY varchar(15),
    DAYS_CREDIT smallint,
    CREDIT_DAY_OVERDUE smallint,
    DAYS_CREDIT_ENDDATE real,
    DAYS_ENDDATE_FACT real,
    AMT_CREDIT_MAX_OVERDUE real,
    CNT_CREDIT_PROLONG smallint,
    AMT_CREDIT_SUM real,
    AMT_CREDIT_SUM_DEBT real,
    AMT_CREDIT_SUM_LIMIT real,
    AMT_CREDIT_SUM_OVERDUE real,
    CREDIT_TYPE varchar(49),
    DAYS_CREDIT_UPDATE integer,
    AMT_ANNUITY real
);
"""

db.execute_query(sql_schema_query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [30]:
from pathlib import Path

bureau_columns = [
    "SK_ID_CURR",
    "SK_ID_BUREAU",
    "CREDIT_ACTIVE",
    "CREDIT_CURRENCY",
    "DAYS_CREDIT",
    "CREDIT_DAY_OVERDUE",
    "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT",
    "AMT_CREDIT_MAX_OVERDUE",
    "CNT_CREDIT_PROLONG",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_OVERDUE",
    "CREDIT_TYPE",
    "DAYS_CREDIT_UPDATE",
    "AMT_ANNUITY"
]

bureau_file = Path(DATA_FULL_PATH) / "bureau.csv"

db.copy_from_csv(
    table_name="bureau",
    file_path=str(bureau_file),
    columns=bureau_columns
)

Данные из bureau.csv успешно загружены в bureau
Время выполнения: 0:00:05


In [7]:
sql_query = """
    SELECT * FROM bureau
    LIMIT 10
"""

df = db.get_df(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,sk_id_bureau,credit_active,credit_currency,days_credit,credit_day_overdue,days_credit_enddate,days_enddate_fact,amt_credit_max_overdue,cnt_credit_prolong,amt_credit_sum,amt_credit_sum_debt,amt_credit_sum_limit,amt_credit_sum_overdue,credit_type,days_credit_update,amt_annuity
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.00,NaN,0.0,Consumer credit,-131,None
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.00,NaN,0.0,Credit card,-20,None
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,None
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,None
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,None
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0,180000.00,71017.38,108982.62,0.0,Credit card,-31,None
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0,42103.80,42103.80,0.00,0.0,Consumer credit,-22,None
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0,76878.45,0.00,0.00,0.0,Consumer credit,-1710,None
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0,103007.70,0.00,0.00,0.0,Consumer credit,-840,None
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0,4500.00,0.00,0.00,0.0,Credit card,-690,None


In [9]:
query = """
SELECT 
    sk_id_curr,
    sk_id_bureau,
    credit_active,
    credit_currency
FROM 
    bureau
"""
db.get_df(query)

Время выполнения: 0:00:02


,sk_id_curr,sk_id_bureau,credit_active,credit_currency
0,215354,5714462,Closed,currency 1
1,215354,5714463,Active,currency 1
2,215354,5714464,Active,currency 1
3,215354,5714465,Active,currency 1
4,215354,5714466,Active,currency 1
...,...,...,...,...
1716423,259355,5057750,Active,currency 1
1716424,100044,5057754,Closed,currency 1
1716425,100044,5057762,Closed,currency 1
1716426,246829,5057770,Closed,currency 1


In [10]:
query = """
    SELECT 
        sk_id_curr AS id_loan,
        COUNT(sk_id_bureau) AS count_loans_bki_history,
        SUM(amt_credit_sum) AS loans_sum
    FROM 
        bureau
    GROUP BY 
        sk_id_curr
    LIMIT 10
"""
db.get_df(query)

Время выполнения: 0:00:00


,id_loan,count_loans_bki_history,loans_sum
0,100001,7,1453365.00
1,100002,8,865055.56
2,100003,4,1017400.50
3,100004,2,189037.80
4,100005,3,657126.00
5,100007,1,146250.00
6,100008,3,468445.50
7,100009,18,4800811.50
8,100010,2,990000.00
9,100011,4,435228.30


### Первичный анализ типов данных

Функция `analyze_dataframe()` выполняет автоматический анализ структуры датафрейма и формирует отчет по каждому столбцу: тип данных, количество пропусков, диапазон значений, длину текстовых полей и проверку на целочисленный формат.


In [6]:
import pandas as pd
from pathlib import Path


def analyze_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in df.columns:
        series = df[col]
        non_null = series.dropna()

        pandas_dtype = str(series.dtype)
        null_count = int(series.isna().sum())
        unique_count = int(non_null.nunique())

        min_value = None
        max_value = None
        max_len = None
        sample_value = None
        is_integer_like = False

        if not non_null.empty:
            sample_value = non_null.iloc[0]

            if pd.api.types.is_numeric_dtype(series):
                min_value = non_null.min()
                max_value = non_null.max()

                # если float, но все значения целые по смыслу
                if pd.api.types.is_float_dtype(series):
                    is_integer_like = ((non_null % 1) == 0).all()
                elif pd.api.types.is_integer_dtype(series):
                    is_integer_like = True

            else:
                max_len = non_null.astype(str).map(len).max()

        rows.append({
            "column_name": col,
            "pandas_dtype": pandas_dtype,
            "null_count": null_count,
            "unique_count": unique_count,
            "min_value": min_value,
            "max_value": max_value,
            "max_len": max_len,
            "is_integer_like": is_integer_like,
            "sample_value": sample_value,
        })

    return pd.DataFrame(rows)

### Автоматический подбор типов данных PostgreSQL

Для автоматизации создания SQL-схемы была реализована функция `suggest_postgres_type()`.

Функция на основе результатов предварительного анализа столбцов предлагает наиболее подходящий тип данных PostgreSQL.

Логика выбора типа:

- для ключевых идентификаторов (`SK_ID_CURR`, `SK_ID_BUREAU`, `SK_ID_PREV`) всегда используется тип `INTEGER`, чтобы обеспечить единый формат связей между таблицами
- для целочисленных признаков автоматически подбирается один из типов:
  - `SMALLINT`
  - `INTEGER`
  - `BIGINT`
  
  выбор зависит от диапазона значений
- для вещественных признаков:
  - если значения по смыслу являются целыми (`12.0`, `45.0`), также подбирается целочисленный SQL-тип
  - в остальных случаях используется `REAL`
- для логических признаков используется `BOOLEAN`
- для строковых признаков используется `VARCHAR(n)`, где `n` — максимальная длина значения в столбце
- если тип определить невозможно, используется `TEXT`


In [7]:
def suggest_postgres_type(column_name: str, pandas_dtype: str, min_value, max_value, max_len, is_integer_like: bool):
    key_columns = {
        "SK_ID_CURR": "INTEGER",
        "SK_ID_BUREAU": "INTEGER",
        "SK_ID_PREV": "INTEGER",
    }

    if column_name in key_columns:
        return key_columns[column_name]

    if pandas_dtype in ["int64", "int32"]:
        if min_value is not None and max_value is not None:
            if -32768 <= min_value <= 32767 and -32768 <= max_value <= 32767:
                return "SMALLINT"
            elif -2147483648 <= min_value <= 2147483647 and -2147483648 <= max_value <= 2147483647:
                return "INTEGER"
            else:
                return "BIGINT"

    if pandas_dtype in ["float64", "float32"]:
        if is_integer_like and min_value is not None and max_value is not None:
            if -32768 <= min_value <= 32767 and -32768 <= max_value <= 32767:
                return "SMALLINT"
            elif -2147483648 <= min_value <= 2147483647 and -2147483648 <= max_value <= 2147483647:
                return "INTEGER"
            else:
                return "BIGINT"

        return "REAL"

    if pandas_dtype == "bool":
        return "BOOLEAN"

    if max_len is not None:
        return f"VARCHAR({int(max_len)})"

    return "TEXT"

### Построение отчета по типам данных

Функция `build_type_report()` формирует итоговый отчет по структуре CSV-файла: выполняет анализ столбцов и автоматически подбирает рекомендуемые типы данных PostgreSQL.


In [8]:
def build_type_report(csv_path: str, nrows: int | None = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path, nrows=nrows)
    report = analyze_dataframe(df)

    report["suggested_pg_type"] = report.apply(
        lambda row: suggest_postgres_type(
            column_name=row["column_name"],
            pandas_dtype=row["pandas_dtype"],
            min_value=row["min_value"],
            max_value=row["max_value"],
            max_len=row["max_len"],
            is_integer_like=row["is_integer_like"],
        ),
        axis=1
    )

    return report

In [13]:
file_path = Path(DATA_FULL_PATH) / "previous_application.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_PREV,int64,0,100000,1.000001e+06,2.845377e+06,NaN,True,2030495,INTEGER
1,SK_ID_CURR,int64,0,79977,1.000060e+05,4.562540e+05,NaN,True,271877,INTEGER
2,NAME_CONTRACT_TYPE,object,0,4,NaN,NaN,15.0,False,Consumer loans,VARCHAR(15)
3,AMT_ANNUITY,float64,21133,52299,0.000000e+00,4.179276e+05,NaN,False,1730.43,REAL
4,AMT_APPLICATION,float64,0,21832,0.000000e+00,3.826372e+06,NaN,False,17145.0,REAL
5,AMT_CREDIT,float64,0,28993,0.000000e+00,4.104351e+06,NaN,False,17145.0,REAL
6,AMT_DOWN_PAYMENT,float64,50584,6409,0.000000e+00,1.035000e+06,NaN,False,0.0,REAL
7,AMT_GOODS_PRICE,float64,21600,21832,0.000000e+00,3.826372e+06,NaN,False,17145.0,REAL
8,WEEKDAY_APPR_PROCESS_START,object,0,7,NaN,NaN,9.0,False,SATURDAY,VARCHAR(9)
9,HOUR_APPR_PROCESS_START,int64,0,24,0.000000e+00,2.300000e+01,NaN,True,15,SMALLINT


In [16]:
file_path = Path(DATA_FULL_PATH) / "POS_CASH_balance.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_PREV,int64,0,90062,1000001.0,2843487.0,NaN,True,1803195,INTEGER
1,SK_ID_CURR,int64,0,77469,100007.0,456255.0,NaN,True,182943,INTEGER
2,MONTHS_BALANCE,int64,0,93,-96.0,-1.0,NaN,True,-31,SMALLINT
3,CNT_INSTALMENT,float64,245,55,1.0,72.0,NaN,True,48.0,SMALLINT
4,CNT_INSTALMENT_FUTURE,float64,245,63,0.0,68.0,NaN,True,45.0,SMALLINT
5,NAME_CONTRACT_STATUS,object,0,6,NaN,NaN,21.0,False,Active,VARCHAR(21)
6,SK_DPD,int64,0,100,0.0,2214.0,NaN,True,0,SMALLINT
7,SK_DPD_DEF,int64,0,39,0.0,203.0,NaN,True,0,SMALLINT


In [18]:
file_path = Path(DATA_FULL_PATH) / "installments_payments.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_PREV,int64,0,71415,1000011.0,2843477.00,None,True,1054186.00,INTEGER
1,SK_ID_CURR,int64,0,48591,100003.0,199999.00,None,True,161674.00,INTEGER
2,NUM_INSTALMENT_VERSION,float64,0,33,0.0,34.00,None,True,1.00,SMALLINT
3,NUM_INSTALMENT_NUMBER,int64,0,211,1.0,237.00,None,True,6.00,SMALLINT
4,DAYS_INSTALMENT,float64,0,2921,-2922.0,-2.00,None,True,-1180.00,SMALLINT
5,DAYS_ENTRY_PAYMENT,float64,0,2948,-3026.0,-2.00,None,True,-1187.00,SMALLINT
6,AMT_INSTALMENT,float64,0,58765,0.0,2292736.86,None,False,6948.36,REAL
7,AMT_PAYMENT,float64,0,60248,0.0,2292736.86,None,False,6948.36,REAL


In [19]:
file_path = Path(DATA_FULL_PATH) / "credit_card_balance.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_PREV,int64,0,53559,1000094.000,2843477.000,NaN,True,2562384,INTEGER
1,SK_ID_CURR,int64,0,53383,100011.000,456248.000,NaN,True,378907,INTEGER
2,MONTHS_BALANCE,int64,0,96,-96.000,-1.000,NaN,True,-6,SMALLINT
3,AMT_BALANCE,float64,0,38312,-135359.010,959924.475,NaN,False,56.97,REAL
4,AMT_CREDIT_LIMIT_ACTUAL,int64,0,135,0.000,1350000.000,NaN,True,135000,INTEGER
5,AMT_DRAWINGS_ATM_CURRENT,float64,22233,430,0.000,1305000.000,NaN,False,0.0,REAL
6,AMT_DRAWINGS_CURRENT,float64,0,6246,0.000,1440180.000,NaN,False,877.5,REAL
7,AMT_DRAWINGS_OTHER_CURRENT,float64,22233,156,0.000,761940.000,NaN,False,0.0,REAL
8,AMT_DRAWINGS_POS_CURRENT,float64,22233,5603,0.000,1440180.000,NaN,False,877.5,REAL
9,AMT_INST_MIN_REGULARITY,float64,6131,12610,0.000,47251.305,NaN,False,1700.325,REAL


In [20]:
file_path = Path(DATA_FULL_PATH) / "bureau_balance.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_BUREAU,int64,0,3111,5001726.0,6560309.0,NaN,True,5715448,INTEGER
1,MONTHS_BALANCE,int64,0,97,-96.0,0.0,NaN,True,0,SMALLINT
2,STATUS,object,0,8,NaN,NaN,1.0,False,C,VARCHAR(1)


In [38]:
file_path = Path(DATA_FULL_PATH) / "application_test.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_CURR,int64,0,48744,100001.000000,4.562500e+05,NaN,True,100001,INTEGER
1,NAME_CONTRACT_TYPE,object,0,2,NaN,NaN,15.0,False,Cash loans,VARCHAR(15)
2,CODE_GENDER,object,0,2,NaN,NaN,1.0,False,F,VARCHAR(1)
3,FLAG_OWN_CAR,object,0,2,NaN,NaN,1.0,False,N,VARCHAR(1)
4,FLAG_OWN_REALTY,object,0,2,NaN,NaN,1.0,False,Y,VARCHAR(1)
5,CNT_CHILDREN,int64,0,11,0.000000,2.000000e+01,NaN,True,0,SMALLINT
6,AMT_INCOME_TOTAL,float64,0,606,26941.500000,4.410000e+06,NaN,False,135000.0,REAL
7,AMT_CREDIT,float64,0,2937,45000.000000,2.245500e+06,NaN,False,568800.0,REAL
8,AMT_ANNUITY,float64,24,7491,2295.000000,1.805760e+05,NaN,False,20560.5,REAL
9,AMT_GOODS_PRICE,float64,0,677,45000.000000,2.245500e+06,NaN,False,450000.0,REAL


In [36]:
file_path = Path(DATA_FULL_PATH) / "application_train.csv"
report_prev = build_type_report(file_path, nrows=100000)
report_prev

,column_name,pandas_dtype,null_count,unique_count,min_value,max_value,max_len,is_integer_like,sample_value,suggested_pg_type
0,SK_ID_CURR,int64,0,100000,1.000020e+05,2.160900e+05,NaN,True,100002,INTEGER
1,TARGET,int64,0,2,0.000000e+00,1.000000e+00,NaN,True,1,SMALLINT
2,NAME_CONTRACT_TYPE,object,0,2,NaN,NaN,15.0,False,Cash loans,VARCHAR(15)
3,CODE_GENDER,object,0,3,NaN,NaN,3.0,False,M,VARCHAR(3)
4,FLAG_OWN_CAR,object,0,2,NaN,NaN,1.0,False,N,VARCHAR(1)
5,FLAG_OWN_REALTY,object,0,2,NaN,NaN,1.0,False,Y,VARCHAR(1)
6,CNT_CHILDREN,int64,0,12,0.000000e+00,1.200000e+01,NaN,True,0,SMALLINT
7,AMT_INCOME_TOTAL,float64,0,1212,2.565000e+04,1.170000e+08,NaN,False,202500.0,REAL
8,AMT_CREDIT,float64,0,4162,4.500000e+04,4.050000e+06,NaN,False,406597.5,REAL
9,AMT_ANNUITY,float64,7,10773,1.980000e+03,2.580255e+05,NaN,False,24700.5,REAL


### Обоснование выбора типов для таблицы `bureau_balance`


`SK_ID_BUREAU` сохранён как `INTEGER`, так как это идентификатор записи бюро, который используется для связи с таблицей `bureau`. Тип согласован с уже созданной таблицей `bureau`, где поле `SK_ID_BUREAU` также имеет тип `INTEGER`.

`MONTHS_BALANCE` сохранён как `SMALLINT`, поскольку столбец содержит только целые значения в диапазоне от `-96` до `0`, что полностью помещается в диапазон `SMALLINT`.

`STATUS` сохранён как `VARCHAR(1)`, так как это категориальный признак, представленный строками длиной 1 символ.

In [9]:
sql_schema_query = """
DROP TABLE IF EXISTS bureau_balance;

CREATE TABLE bureau_balance (
    SK_ID_BUREAU INTEGER,
    MONTHS_BALANCE SMALLINT,
    STATUS VARCHAR(1)
);
"""

db.execute_query(sql_schema_query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [10]:


bureau_balance_columns = [
    "SK_ID_BUREAU",
    "MONTHS_BALANCE",
    "STATUS"
]

bureau_balance_file = Path(DATA_FULL_PATH) / "bureau_balance.csv"

db.copy_from_csv(
    table_name="bureau_balance",
    file_path=str(bureau_balance_file),
    columns=bureau_balance_columns
)

Данные из bureau_balance.csv успешно загружены в bureau_balance
Время выполнения: 0:00:17


In [8]:
db.get_df("SELECT * FROM bureau_balance LIMIT 10;")

Время выполнения: 0:00:00


,sk_id_bureau,months_balance,status
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
5,5715448,-5,C
6,5715448,-6,C
7,5715448,-7,C
8,5715448,-8,C
9,5715448,-9,0


In [11]:
sql_schema_query = """
DROP TABLE IF EXISTS previous_application;

CREATE TABLE previous_application(
    SK_ID_PREV INTEGER,
    SK_ID_CURR INTEGER,
    NAME_CONTRACT_TYPE VARCHAR(15),
    AMT_ANNUITY REAL,
    AMT_APPLICATION REAL,
    AMT_CREDIT REAL,
    AMT_DOWN_PAYMENT REAL,
    AMT_GOODS_PRICE REAL,
    WEEKDAY_APPR_PROCESS_START VARCHAR(9),
    HOUR_APPR_PROCESS_START SMALLINT,
    FLAG_LAST_APPL_PER_CONTRACT VARCHAR(1),
    NFLAG_LAST_APPL_IN_DAY SMALLINT,
    RATE_DOWN_PAYMENT REAL,
    RATE_INTEREST_PRIMARY REAL,
    RATE_INTEREST_PRIVILEGED REAL,
    NAME_CASH_LOAN_PURPOSE VARCHAR(32),
    NAME_CONTRACT_STATUS VARCHAR(12),
    DAYS_DECISION SMALLINT,
    NAME_PAYMENT_TYPE VARCHAR(41),
    CODE_REJECT_REASON VARCHAR(6),
    NAME_TYPE_SUITE VARCHAR(15),
    NAME_CLIENT_TYPE VARCHAR(9),
    NAME_GOODS_CATEGORY VARCHAR(24),
    NAME_PORTFOLIO VARCHAR(5),
    NAME_PRODUCT_TYPE VARCHAR(7),
    CHANNEL_TYPE VARCHAR(26),
    SELLERPLACE_AREA INTEGER,
    NAME_SELLER_INDUSTRY VARCHAR(20),
    CNT_PAYMENT REAL,
    NAME_YIELD_GROUP VARCHAR(10),
    PRODUCT_COMBINATION VARCHAR(30),
    DAYS_FIRST_DRAWING REAL,
    DAYS_FIRST_DUE REAL,
    DAYS_LAST_DUE_1ST_VERSION REAL,
    DAYS_LAST_DUE REAL,
    DAYS_TERMINATION REAL,
    NFLAG_INSURED_ON_APPROVAL REAL
);
"""
db.execute_query(sql_schema_query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [12]:
previous_application_columns = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "NAME_CONTRACT_TYPE",
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "WEEKDAY_APPR_PROCESS_START",
    "HOUR_APPR_PROCESS_START",
    "FLAG_LAST_APPL_PER_CONTRACT",
    "NFLAG_LAST_APPL_IN_DAY",
    "RATE_DOWN_PAYMENT",
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED",
    "NAME_CASH_LOAN_PURPOSE",
    "NAME_CONTRACT_STATUS",
    "DAYS_DECISION",
    "NAME_PAYMENT_TYPE",
    "CODE_REJECT_REASON",
    "NAME_TYPE_SUITE",
    "NAME_CLIENT_TYPE",
    "NAME_GOODS_CATEGORY",
    "NAME_PORTFOLIO",
    "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE",
    "SELLERPLACE_AREA",
    "NAME_SELLER_INDUSTRY",
    "CNT_PAYMENT",
    "NAME_YIELD_GROUP",
    "PRODUCT_COMBINATION",
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
    "NFLAG_INSURED_ON_APPROVAL"
]
from pathlib import Path

previous_application_file = Path(DATA_FULL_PATH) / "previous_application.csv"

db.copy_from_csv(
    table_name="previous_application",
    file_path=str(previous_application_file),
    columns=previous_application_columns
)

Данные из previous_application.csv успешно загружены в previous_application
Время выполнения: 0:00:12


### Комментарий по выбору типов при загрузке `previous_application`

Для таблицы `previous_application` при прямой загрузке CSV через `COPY` часть столбцов была сохранена как `REAL`, хотя по смыслу некоторые из них являются целочисленными признаками (`CNT_PAYMENT`, `DAYS_FIRST_DRAWING`, `DAYS_FIRST_DUE`, `DAYS_LAST_DUE`, `DAYS_TERMINATION`, `NFLAG_INSURED_ON_APPROVAL`).

Это связано с тем, что в исходном CSV-файле такие значения записаны в формате с десятичной точкой, например `12.0` или `-42.0`. PostgreSQL не может напрямую загрузить такие значения в столбцы типа `SMALLINT` или `INTEGER` через `COPY`.

Поэтому на этапе первичной загрузки данных был выбран тип `REAL`, соответствующий физическому формату данных в CSV. При этом по смыслу эти признаки могут рассматриваться как целочисленные и при необходимости быть приведены к целочисленному типу на последующих этапах обработки данных.

In [13]:
sql_schema_query = """
DROP TABLE IF EXISTS pos_cash_balance;

CREATE TABLE pos_cash_balance(
    SK_ID_PREV INTEGER,
    SK_ID_CURR INTEGER,
    MONTHS_BALANCE SMALLINT,
    CNT_INSTALMENT REAL,
    CNT_INSTALMENT_FUTURE REAL,
    NAME_CONTRACT_STATUS VARCHAR(21),
    SK_DPD SMALLINT,
    SK_DPD_DEF SMALLINT
);
"""
db.execute_query(sql_schema_query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [14]:
pos_cash_balance_columns = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "MONTHS_BALANCE",
    "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE",
    "NAME_CONTRACT_STATUS",
    "SK_DPD",
    "SK_DPD_DEF"
]
from pathlib import Path

pos_cash_balance_file = Path(DATA_FULL_PATH) / "POS_CASH_balance.csv"

db.copy_from_csv(
    table_name="pos_cash_balance",
    file_path=str(pos_cash_balance_file),
    columns=pos_cash_balance_columns
)

Данные из POS_CASH_balance.csv успешно загружены в pos_cash_balance
Время выполнения: 0:00:17


In [26]:
db.get_df("SELECT * FROM pos_cash_balance LIMIT 10;")

Время выполнения: 0:00:00


,sk_id_prev,sk_id_curr,months_balance,cnt_instalment,cnt_instalment_future,name_contract_status,sk_dpd,sk_dpd_def
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0
5,2207092,342166,-32,12.0,12.0,Active,0,0
6,1110516,204376,-38,48.0,43.0,Active,0,0
7,1387235,153211,-35,36.0,36.0,Active,0,0
8,1220500,112740,-31,12.0,12.0,Active,0,0
9,2371489,274851,-32,24.0,16.0,Active,0,0


### Обоснование выбора типов для таблицы `pos_cash_balance`


Поля `SK_ID_PREV` и `SK_ID_CURR` сохранены как `INTEGER`, так как это идентификаторы, используемые для связи с другими таблицами. Типы согласованы с уже созданными таблицами `previous_application`, `bureau` и другими таблицами проекта.

Поле `MONTHS_BALANCE` сохранено как `SMALLINT`, поскольку содержит целые значения в небольшом диапазоне от `-96` до `-1`.

Поля `CNT_INSTALMENT` и `CNT_INSTALMENT_FUTURE` по смыслу являются счетчиками и могли бы быть сохранены как целочисленные типы. Однако в исходном CSV-файле они представлены как значения формата `48.0`, `45.0`, то есть как числа с плавающей точкой. Поэтому для корректной прямой загрузки через `COPY` эти поля сохранены как `REAL`.

Поле `NAME_CONTRACT_STATUS` сохранено как `VARCHAR(21)`, так как это категориальный строковый признак, а максимальная длина значения в столбце составляет 21 символ.

Поля `SK_DPD` и `SK_DPD_DEF` сохранены как `SMALLINT`, поскольку содержат небольшие целочисленные значения и не имеют пропусков.

In [15]:
sql_schema_query = """
DROP TABLE IF EXISTS installments_payments;

CREATE TABLE installments_payments(
    SK_ID_PREV INTEGER,
    SK_ID_CURR INTEGER,
    NUM_INSTALMENT_VERSION REAL,
    NUM_INSTALMENT_NUMBER SMALLINT,
    DAYS_INSTALMENT REAL,
    DAYS_ENTRY_PAYMENT REAL,
    AMT_INSTALMENT REAL,
    AMT_PAYMENT REAL
);
"""
db.execute_query(sql_schema_query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [16]:
installments_payments_columns = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT",
    "AMT_PAYMENT"
]
from pathlib import Path

installments_payments_file = Path(DATA_FULL_PATH) / "installments_payments.csv"

db.copy_from_csv(
    table_name="installments_payments",
    file_path=str(installments_payments_file),
    columns=installments_payments_columns
)

Данные из installments_payments.csv успешно загружены в installments_payments
Время выполнения: 0:00:26


### Обоснование выбора типов для таблицы `installments_payments`


Поля `SK_ID_PREV` и `SK_ID_CURR` сохранены как `INTEGER`, так как это идентификаторы, используемые для связи с другими таблицами. Типы согласованы с ранее созданными таблицами.

Поле `NUM_INSTALMENT_NUMBER` сохранено как `SMALLINT`, поскольку содержит небольшие целочисленные значения в диапазоне от 1 до 237.

Поля `NUM_INSTALMENT_VERSION`, `DAYS_INSTALMENT` и `DAYS_ENTRY_PAYMENT` по смыслу являются целочисленными признаками, однако в исходном CSV-файле они представлены в формате чисел с плавающей точкой, например `1.0` или `-1180.0`. Поэтому для корректной прямой загрузки данных через `COPY` эти поля сохранены как `REAL`.

Поля `AMT_INSTALMENT` и `AMT_PAYMENT` сохранены как `REAL`, поскольку содержат вещественные значения денежных сумм.

In [17]:
sql_schema_query = """
DROP TABLE IF EXISTS credit_card_balance;

CREATE TABLE credit_card_balance(
    SK_ID_PREV INTEGER,
    SK_ID_CURR INTEGER,
    MONTHS_BALANCE SMALLINT,
    AMT_BALANCE REAL,
    AMT_CREDIT_LIMIT_ACTUAL INTEGER,
    AMT_DRAWINGS_ATM_CURRENT REAL,
    AMT_DRAWINGS_CURRENT REAL,
    AMT_DRAWINGS_OTHER_CURRENT REAL,
    AMT_DRAWINGS_POS_CURRENT REAL,
    AMT_INST_MIN_REGULARITY REAL,
    AMT_PAYMENT_CURRENT REAL,
    AMT_PAYMENT_TOTAL_CURRENT REAL,
    AMT_RECEIVABLE_PRINCIPAL REAL,
    AMT_RECIVABLE REAL,
    AMT_TOTAL_RECEIVABLE REAL,
    CNT_DRAWINGS_ATM_CURRENT REAL,
    CNT_DRAWINGS_CURRENT SMALLINT,
    CNT_DRAWINGS_OTHER_CURRENT REAL,
    CNT_DRAWINGS_POS_CURRENT REAL,
    CNT_INSTALMENT_MATURE_CUM REAL,
    NAME_CONTRACT_STATUS VARCHAR(13),
    SK_DPD SMALLINT,
    SK_DPD_DEF SMALLINT
);
"""
db.execute_query(sql_schema_query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [18]:
credit_card_balance_columns = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "MONTHS_BALANCE",
    "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_ATM_CURRENT",
    "AMT_DRAWINGS_CURRENT",
    "AMT_DRAWINGS_OTHER_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    "AMT_INST_MIN_REGULARITY",
    "AMT_PAYMENT_CURRENT",
    "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL",
    "AMT_RECIVABLE",
    "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_ATM_CURRENT",
    "CNT_DRAWINGS_CURRENT",
    "CNT_DRAWINGS_OTHER_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT",
    "CNT_INSTALMENT_MATURE_CUM",
    "NAME_CONTRACT_STATUS",
    "SK_DPD",
    "SK_DPD_DEF"
]

from pathlib import Path

credit_card_balance_file = Path(DATA_FULL_PATH) / "credit_card_balance.csv"

db.copy_from_csv(
    table_name="credit_card_balance",
    file_path=str(credit_card_balance_file),
    columns=credit_card_balance_columns
)

Данные из credit_card_balance.csv успешно загружены в credit_card_balance
Время выполнения: 0:00:16


### Обоснование выбора типов для таблицы `credit_card_balance`


Поля `SK_ID_PREV` и `SK_ID_CURR` сохранены как `INTEGER`, поскольку это идентификаторы, используемые для связи с другими таблицами. Типы согласованы с уже загруженными таблицами проекта.

Поле `MONTHS_BALANCE` сохранено как `SMALLINT`, так как содержит небольшие целочисленные значения в диапазоне от `-96` до `-1`.

Поле `AMT_CREDIT_LIMIT_ACTUAL` сохранено как `INTEGER`, поскольку это целочисленный денежный лимит без пропусков и без дробной части.

Поля `AMT_*` сохранены как `REAL`, так как содержат вещественные денежные значения.

Поля `CNT_DRAWINGS_ATM_CURRENT`, `CNT_DRAWINGS_OTHER_CURRENT`, `CNT_DRAWINGS_POS_CURRENT`, `CNT_INSTALMENT_MATURE_CUM` по смыслу являются счетчиками, однако в исходном CSV представлены как числа с плавающей точкой (`0.0`, `1.0`, `35.0`). Поэтому для корректной прямой загрузки через `COPY` они сохранены как `REAL`.

Поле `CNT_DRAWINGS_CURRENT` сохранено как `SMALLINT`, так как в исходном файле оно имеет целочисленный тип и небольшой диапазон значений.

Поле `NAME_CONTRACT_STATUS` сохранено как `VARCHAR(13)`, поскольку это категориальный строковый признак, а максимальная длина значения в столбце составляет 13 символов.

Поля `SK_DPD` и `SK_DPD_DEF` сохранены как `SMALLINT`, поскольку содержат небольшие целочисленные значения без пропусков.

In [19]:
application_column_types = {
    "SK_ID_CURR": "INTEGER",
    "TARGET": "SMALLINT",
    "NAME_CONTRACT_TYPE": "VARCHAR(15)",
    "CODE_GENDER": "VARCHAR(3)",
    "FLAG_OWN_CAR": "VARCHAR(1)",
    "FLAG_OWN_REALTY": "VARCHAR(1)",
    "CNT_CHILDREN": "SMALLINT",
    "AMT_INCOME_TOTAL": "REAL",
    "AMT_CREDIT": "REAL",
    "AMT_ANNUITY": "REAL",
    "AMT_GOODS_PRICE": "REAL",
    "NAME_TYPE_SUITE": "VARCHAR(15)",
    "NAME_INCOME_TYPE": "VARCHAR(20)",
    "NAME_EDUCATION_TYPE": "VARCHAR(29)",
    "NAME_FAMILY_STATUS": "VARCHAR(20)",
    "NAME_HOUSING_TYPE": "VARCHAR(19)",
    "REGION_POPULATION_RELATIVE": "REAL",
    "DAYS_BIRTH": "SMALLINT",
    "DAYS_EMPLOYED": "INTEGER",
    "DAYS_REGISTRATION": "REAL",
    "DAYS_ID_PUBLISH": "SMALLINT",
    "OWN_CAR_AGE": "REAL",
    "FLAG_MOBIL": "SMALLINT",
    "FLAG_EMP_PHONE": "SMALLINT",
    "FLAG_WORK_PHONE": "SMALLINT",
    "FLAG_CONT_MOBILE": "SMALLINT",
    "FLAG_PHONE": "SMALLINT",
    "FLAG_EMAIL": "SMALLINT",
    "OCCUPATION_TYPE": "VARCHAR(21)",
    "CNT_FAM_MEMBERS": "REAL",
    "REGION_RATING_CLIENT": "SMALLINT",
    "REGION_RATING_CLIENT_W_CITY": "SMALLINT",
    "WEEKDAY_APPR_PROCESS_START": "VARCHAR(9)",
    "HOUR_APPR_PROCESS_START": "SMALLINT",
    "REG_REGION_NOT_LIVE_REGION": "SMALLINT",
    "REG_REGION_NOT_WORK_REGION": "SMALLINT",
    "LIVE_REGION_NOT_WORK_REGION": "SMALLINT",
    "REG_CITY_NOT_LIVE_CITY": "SMALLINT",
    "REG_CITY_NOT_WORK_CITY": "SMALLINT",
    "LIVE_CITY_NOT_WORK_CITY": "SMALLINT",
    "ORGANIZATION_TYPE": "VARCHAR(22)",
    "EXT_SOURCE_1": "REAL",
    "EXT_SOURCE_2": "REAL",
    "EXT_SOURCE_3": "REAL",
    "APARTMENTS_AVG": "REAL",
    "BASEMENTAREA_AVG": "REAL",
    "YEARS_BEGINEXPLUATATION_AVG": "REAL",
    "YEARS_BUILD_AVG": "REAL",
    "COMMONAREA_AVG": "REAL",
    "ELEVATORS_AVG": "REAL",
    "ENTRANCES_AVG": "REAL",
    "FLOORSMAX_AVG": "REAL",
    "FLOORSMIN_AVG": "REAL",
    "LANDAREA_AVG": "REAL",
    "LIVINGAPARTMENTS_AVG": "REAL",
    "LIVINGAREA_AVG": "REAL",
    "NONLIVINGAPARTMENTS_AVG": "REAL",
    "NONLIVINGAREA_AVG": "REAL",
    "APARTMENTS_MODE": "REAL",
    "BASEMENTAREA_MODE": "REAL",
    "YEARS_BEGINEXPLUATATION_MODE": "REAL",
    "YEARS_BUILD_MODE": "REAL",
    "COMMONAREA_MODE": "REAL",
    "ELEVATORS_MODE": "REAL",
    "ENTRANCES_MODE": "REAL",
    "FLOORSMAX_MODE": "REAL",
    "FLOORSMIN_MODE": "REAL",
    "LANDAREA_MODE": "REAL",
    "LIVINGAPARTMENTS_MODE": "REAL",
    "LIVINGAREA_MODE": "REAL",
    "NONLIVINGAPARTMENTS_MODE": "REAL",
    "NONLIVINGAREA_MODE": "REAL",
    "APARTMENTS_MEDI": "REAL",
    "BASEMENTAREA_MEDI": "REAL",
    "YEARS_BEGINEXPLUATATION_MEDI": "REAL",
    "YEARS_BUILD_MEDI": "REAL",
    "COMMONAREA_MEDI": "REAL",
    "ELEVATORS_MEDI": "REAL",
    "ENTRANCES_MEDI": "REAL",
    "FLOORSMAX_MEDI": "REAL",
    "FLOORSMIN_MEDI": "REAL",
    "LANDAREA_MEDI": "REAL",
    "LIVINGAPARTMENTS_MEDI": "REAL",
    "LIVINGAREA_MEDI": "REAL",
    "NONLIVINGAPARTMENTS_MEDI": "REAL",
    "NONLIVINGAREA_MEDI": "REAL",
    "FONDKAPREMONT_MODE": "VARCHAR(21)",
    "HOUSETYPE_MODE": "VARCHAR(16)",
    "TOTALAREA_MODE": "REAL",
    "WALLSMATERIAL_MODE": "VARCHAR(12)",
    "EMERGENCYSTATE_MODE": "VARCHAR(3)",
    "OBS_30_CNT_SOCIAL_CIRCLE": "REAL",
    "DEF_30_CNT_SOCIAL_CIRCLE": "REAL",
    "OBS_60_CNT_SOCIAL_CIRCLE": "REAL",
    "DEF_60_CNT_SOCIAL_CIRCLE": "REAL",
    "DAYS_LAST_PHONE_CHANGE": "REAL",
    "FLAG_DOCUMENT_2": "SMALLINT",
    "FLAG_DOCUMENT_3": "SMALLINT",
    "FLAG_DOCUMENT_4": "SMALLINT",
    "FLAG_DOCUMENT_5": "SMALLINT",
    "FLAG_DOCUMENT_6": "SMALLINT",
    "FLAG_DOCUMENT_7": "SMALLINT",
    "FLAG_DOCUMENT_8": "SMALLINT",
    "FLAG_DOCUMENT_9": "SMALLINT",
    "FLAG_DOCUMENT_10": "SMALLINT",
    "FLAG_DOCUMENT_11": "SMALLINT",
    "FLAG_DOCUMENT_12": "SMALLINT",
    "FLAG_DOCUMENT_13": "SMALLINT",
    "FLAG_DOCUMENT_14": "SMALLINT",
    "FLAG_DOCUMENT_15": "SMALLINT",
    "FLAG_DOCUMENT_16": "SMALLINT",
    "FLAG_DOCUMENT_17": "SMALLINT",
    "FLAG_DOCUMENT_18": "SMALLINT",
    "FLAG_DOCUMENT_19": "SMALLINT",
    "FLAG_DOCUMENT_20": "SMALLINT",
    "FLAG_DOCUMENT_21": "SMALLINT",
    "AMT_REQ_CREDIT_BUREAU_HOUR": "REAL",
    "AMT_REQ_CREDIT_BUREAU_DAY": "REAL",
    "AMT_REQ_CREDIT_BUREAU_WEEK": "REAL",
    "AMT_REQ_CREDIT_BUREAU_MON": "REAL",
    "AMT_REQ_CREDIT_BUREAU_QRT": "REAL",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "REAL",
    "DATASET_SOURCE": "VARCHAR(5)"
}

In [20]:
def generate_create_table_sql(table_name: str, columns_with_types: dict[str, str]) -> str:
    columns_sql = ",\n    ".join(
        [f"{col} {col_type}" for col, col_type in columns_with_types.items()]
    )

    return f"""
DROP TABLE IF EXISTS {table_name};

CREATE TABLE {table_name}(
    {columns_sql}
);
"""

In [21]:
sql_schema_query = generate_create_table_sql("application", application_column_types)
print(sql_schema_query)
db.execute_query(sql_schema_query)


DROP TABLE IF EXISTS application;

CREATE TABLE application(
    SK_ID_CURR INTEGER,
    TARGET SMALLINT,
    NAME_CONTRACT_TYPE VARCHAR(15),
    CODE_GENDER VARCHAR(3),
    FLAG_OWN_CAR VARCHAR(1),
    FLAG_OWN_REALTY VARCHAR(1),
    CNT_CHILDREN SMALLINT,
    AMT_INCOME_TOTAL REAL,
    AMT_CREDIT REAL,
    AMT_ANNUITY REAL,
    AMT_GOODS_PRICE REAL,
    NAME_TYPE_SUITE VARCHAR(15),
    NAME_INCOME_TYPE VARCHAR(20),
    NAME_EDUCATION_TYPE VARCHAR(29),
    NAME_FAMILY_STATUS VARCHAR(20),
    NAME_HOUSING_TYPE VARCHAR(19),
    REGION_POPULATION_RELATIVE REAL,
    DAYS_BIRTH SMALLINT,
    DAYS_EMPLOYED INTEGER,
    DAYS_REGISTRATION REAL,
    DAYS_ID_PUBLISH SMALLINT,
    OWN_CAR_AGE REAL,
    FLAG_MOBIL SMALLINT,
    FLAG_EMP_PHONE SMALLINT,
    FLAG_WORK_PHONE SMALLINT,
    FLAG_CONT_MOBILE SMALLINT,
    FLAG_PHONE SMALLINT,
    FLAG_EMAIL SMALLINT,
    OCCUPATION_TYPE VARCHAR(21),
    CNT_FAM_MEMBERS REAL,
    REGION_RATING_CLIENT SMALLINT,
    REGION_RATING_CLIENT_W_CITY SMALLINT,


In [22]:
application_train_columns = [col for col in application_column_types.keys() if col != "DATASET_SOURCE"]

In [23]:
application_test_columns = [
    col for col in application_column_types.keys()
    if col not in ("TARGET", "DATASET_SOURCE")
]

In [24]:
db.execute_query("""
UPDATE application
SET DATASET_SOURCE = 'train'
WHERE DATASET_SOURCE IS NULL AND TARGET IS NOT NULL;
""")

Запрос выполнен успешно
Время выполнения: 0:00:00


In [25]:
application_test_file = Path(DATA_FULL_PATH) / "application_test.csv"

db.copy_from_csv(
    table_name="application",
    file_path=str(application_test_file),
    columns=application_test_columns
)

Данные из application_test.csv успешно загружены в application
Время выполнения: 0:00:00


In [26]:
from pathlib import Path

application_train_file = Path(DATA_FULL_PATH) / "application_train.csv"

db.copy_from_csv(
    table_name="application",
    file_path=str(application_train_file),
    columns=application_train_columns
)

Данные из application_train.csv успешно загружены в application
Время выполнения: 0:00:04


In [27]:
db.execute_query("""
UPDATE application
SET DATASET_SOURCE = 'train'
WHERE DATASET_SOURCE IS NULL AND TARGET IS NOT NULL;
""")

Запрос выполнен успешно
Время выполнения: 0:00:05


In [28]:
db.execute_query("""
UPDATE application
SET DATASET_SOURCE = 'test'
WHERE DATASET_SOURCE IS NULL AND TARGET IS NULL;
""")

Запрос выполнен успешно
Время выполнения: 0:00:04


In [18]:
db.get_df("""
SELECT
    DATASET_SOURCE,
    COUNT(*) AS row_count,
    COUNT(TARGET) AS target_not_null_cnt
FROM application
GROUP BY DATASET_SOURCE
ORDER BY DATASET_SOURCE;
""")

Время выполнения: 0:00:00


,dataset_source,row_count,target_not_null_cnt
0,test,48744,0
1,train,307511,307511


### Обоснование выбора структуры таблицы `application`

Таблицы `application_train` и `application_test` были объединены в одну таблицу `application`, так как они имеют одинаковую структуру признаков и различаются только наличием целевой переменной `TARGET` в обучающей выборке.

Поле `TARGET` сохранено в таблице `application` как `SMALLINT` и допускает `NULL`, поскольку для строк из `application_test` значения целевой переменной отсутствуют.

Для удобства различения источника записи добавлено поле `DATASET_SOURCE`, принимающее значения `train` и `test`.

Поле `SK_ID_CURR` сохранено как `INTEGER` и используется как основной идентификатор клиента. Его тип согласован с другими таблицами проекта для последующего задания связей.

Столбцы типа `int64` с небольшим диапазоном значений сохранены как `SMALLINT`, а более широкие целочисленные идентификаторы — как `INTEGER`.

Столбцы типа `float64` сохранены как `REAL`, включая те, которые по смыслу являются целочисленными, поскольку в исходных CSV-файлах они представлены в формате чисел с плавающей точкой (`2.0`, `1.0` и т.п.), и это необходимо для корректной прямой загрузки через `COPY`.

Категориальные строковые признаки сохранены как `VARCHAR(N)`, где `N` соответствует максимальной длине значения в столбце.

In [19]:
query = """
    SELECT *
    FROM
        application
    JOIN
        bureau
        ON application.sk_id_curr = bureau.sk_id_curr
    
"""
db.get_df(query)

Время выполнения: 0:01:29


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,days_enddate_fact,amt_credit_max_overdue,cnt_credit_prolong,amt_credit_sum,amt_credit_sum_debt,amt_credit_sum_limit,amt_credit_sum_overdue,credit_type,days_credit_update,amt_annuity
0,148877,0.0,Cash loans,F,N,Y,0,360000.0,521280.0,26743.5,...,NaN,NaN,0,315000.0,328086.00,NaN,0.0,Credit card,-6,NaN
1,148877,0.0,Cash loans,F,N,Y,0,360000.0,521280.0,26743.5,...,NaN,NaN,0,272520.0,252585.00,0.0,0.0,Consumer credit,-18,9499.500
2,148877,0.0,Cash loans,F,N,Y,0,360000.0,521280.0,26743.5,...,NaN,NaN,0,63427.5,47389.50,0.0,0.0,Consumer credit,-5,37469.387
3,148877,0.0,Cash loans,F,N,Y,0,360000.0,521280.0,26743.5,...,NaN,NaN,0,272520.0,254729.61,NaN,0.0,Consumer credit,-8,9499.500
4,148877,0.0,Cash loans,F,N,Y,0,360000.0,521280.0,26743.5,...,NaN,NaN,0,1836000.0,408431.12,NaN,0.0,Consumer credit,-8,9499.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,453509,NaN,Cash loans,M,Y,Y,0,144000.0,906660.0,36531.0,...,-1620.0,0.0,0,33840.0,0.00,0.0,0.0,Consumer credit,-1620,NaN
1716424,453509,NaN,Cash loans,M,Y,Y,0,144000.0,906660.0,36531.0,...,-348.0,NaN,0,171000.0,0.00,0.0,0.0,Credit card,-345,0.000
1716425,453556,NaN,Cash loans,F,N,Y,0,157500.0,156384.0,16155.0,...,-1631.0,0.0,0,90000.0,NaN,NaN,0.0,Consumer credit,-1630,NaN
1716426,453556,NaN,Cash loans,F,N,Y,0,157500.0,156384.0,16155.0,...,-1098.0,NaN,0,314505.0,0.00,0.0,0.0,Consumer credit,-1079,0.000


In [20]:
query = """
SELECT 
    application.sk_id_curr, 
    code_gender, 
    sk_id_bureau, 
    amt_credit_sum
FROM 
    application
JOIN
    bureau
    ON
    application.sk_id_curr = bureau.sk_id_curr
"""
db.get_df(query)

Время выполнения: 0:00:03


,sk_id_curr,code_gender,sk_id_bureau,amt_credit_sum
0,215354,F,5714462,91323.00
1,215354,F,5714463,225000.00
2,215354,F,5714464,464323.50
3,215354,F,5714465,90000.00
4,215354,F,5714466,2700000.00
...,...,...,...,...
1716423,318662,F,5057561,147150.00
1716424,145715,F,5057569,35032.50
1716425,317217,F,5057684,93955.50
1716426,352790,M,5057718,28248.84


In [21]:
query = """
SELECT
    app.sk_id_curr,
    app.code_gender,
    bureau_new.count_loans_bki_history,
    bureau_new.loans_sum
FROM application AS app
JOIN (
    SELECT
        sk_id_curr,
        COUNT(sk_id_bureau) AS count_loans_bki_history,
        SUM(amt_credit_sum) AS loans_sum
    FROM bureau
    GROUP BY sk_id_curr
) AS bureau_new
    ON app.sk_id_curr = bureau_new.sk_id_curr
"""
db.get_df(query)

Время выполнения: 0:00:01


,sk_id_curr,code_gender,count_loans_bki_history,loans_sum
0,104120,F,2,378000.0
1,107395,F,3,333000.0
2,119861,F,5,899878.5
3,126553,F,20,4428844.0
4,128433,F,6,782176.5
...,...,...,...,...
305806,456168,F,2,1962000.0
305807,456170,F,3,361381.5
305808,456189,F,6,1905309.0
305809,456223,F,5,2156309.5


### Назначение ключей и индексов


In [5]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT SK_ID_CURR) AS distinct_sk_id_curr
FROM application;
"""
db.get_df(query)

Время выполнения: 0:00:01


,total_rows,distinct_sk_id_curr
0,356255,356255


In [6]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT SK_ID_BUREAU) AS distinct_sk_id_bureau
FROM bureau;
"""
db.get_df(query)

Время выполнения: 0:00:00


,total_rows,distinct_sk_id_bureau
0,1716428,1716428


In [7]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT SK_ID_PREV) AS distinct_sk_id_prev
FROM previous_application;
"""
db.get_df(query)

Время выполнения: 0:00:00


,total_rows,distinct_sk_id_prev
0,1670214,1670214


In [8]:
query = """
SELECT COUNT(*) AS duplicate_cnt
FROM (
    SELECT SK_ID_BUREAU, MONTHS_BALANCE, COUNT(*)
    FROM bureau_balance
    GROUP BY SK_ID_BUREAU, MONTHS_BALANCE
    HAVING COUNT(*) > 1
) t;
"""
db.get_df(query)

Время выполнения: 0:00:31


,duplicate_cnt
0,0


In [9]:
query = """
SELECT COUNT(*) AS duplicate_cnt
FROM (
    SELECT SK_ID_PREV, MONTHS_BALANCE, COUNT(*)
    FROM pos_cash_balance
    GROUP BY SK_ID_PREV, MONTHS_BALANCE
    HAVING COUNT(*) > 1
) t;
"""
db.get_df(query)

Время выполнения: 0:00:07


,duplicate_cnt
0,0


In [10]:
query = """
SELECT COUNT(*) AS duplicate_cnt
FROM (
    SELECT SK_ID_PREV, MONTHS_BALANCE, COUNT(*)
    FROM credit_card_balance
    GROUP BY SK_ID_PREV, MONTHS_BALANCE
    HAVING COUNT(*) > 1
) t;
"""
db.get_df(query)

Время выполнения: 0:00:03


,duplicate_cnt
0,0


In [11]:
query = """
SELECT COUNT(*) AS duplicate_cnt
FROM (
    SELECT
        SK_ID_PREV,
        NUM_INSTALMENT_NUMBER,
        DAYS_INSTALMENT,
        COUNT(*)
    FROM installments_payments
    GROUP BY
        SK_ID_PREV,
        NUM_INSTALMENT_NUMBER,
        DAYS_INSTALMENT
    HAVING COUNT(*) > 1
) t;
"""
db.get_df(query)

Время выполнения: 0:00:13


,duplicate_cnt
0,730392


Если есть дубликаты, лучше не делать PK, а оставить только индексы.

In [ ]:
query = """
ALTER TABLE application
ADD CONSTRAINT pk_application
PRIMARY KEY (SK_ID_CURR);
"""
db.execute_query(query)

In [16]:
query = """
ALTER TABLE bureau
ADD CONSTRAINT pk_bureau
PRIMARY KEY (SK_ID_BUREAU);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [17]:
query = """
ALTER TABLE previous_application
ADD CONSTRAINT pk_previous_application
PRIMARY KEY (SK_ID_PREV);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:01


In [18]:
query = """
ALTER TABLE bureau_balance
ADD CONSTRAINT pk_bureau_balance
PRIMARY KEY (SK_ID_BUREAU, MONTHS_BALANCE);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:21


In [19]:
query = """
ALTER TABLE pos_cash_balance
ADD CONSTRAINT pk_pos_cash_balance
PRIMARY KEY (SK_ID_PREV, MONTHS_BALANCE);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:06


In [20]:
query = """
ALTER TABLE credit_card_balance
ADD CONSTRAINT pk_credit_card_balance
PRIMARY KEY (SK_ID_PREV, MONTHS_BALANCE);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:03


In [27]:
query = """
ALTER TABLE bureau
ADD CONSTRAINT fk_bureau_application
FOREIGN KEY (SK_ID_CURR) REFERENCES application(SK_ID_CURR);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:00


In [ ]:
query = """
ALTER TABLE previous_application
ADD CONSTRAINT fk_previous_application_application
FOREIGN KEY (SK_ID_CURR) REFERENCES application(SK_ID_CURR);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:00


### Проверка внешних ключей

Перед назначением внешних ключей была выполнена проверка ссылочной целостности между дочерними и родительскими таблицами.

Для связей:
- `bureau.SK_ID_CURR -> application.SK_ID_CURR`
- `previous_application.SK_ID_CURR -> application.SK_ID_CURR`

нарушений не обнаружено, поэтому внешние ключи были добавлены.

Для связей:
- `pos_cash_balance.SK_ID_PREV -> previous_application.SK_ID_PREV`
- `credit_card_balance.SK_ID_PREV -> previous_application.SK_ID_PREV`
- `installments_payments.SK_ID_PREV -> previous_application.SK_ID_PREV`
- `bureau_balance.SK_ID_BUREAU -> bureau.SK_ID_BUREAU`

были обнаружены значения `SK_ID_PREV`, отсутствующие в таблице `previous_application`. Поэтому внешние ключи для этих связей не назначались. Вместо этого были созданы индексы по соответствующим полям для ускорения соединений таблиц.

In [34]:
query = """
CREATE INDEX IF NOT EXISTS idx_pos_cash_balance_sk_id_prev
ON pos_cash_balance(SK_ID_PREV);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:04


In [35]:
query = """
CREATE INDEX IF NOT EXISTS idx_credit_card_balance_sk_id_prev
ON credit_card_balance(SK_ID_PREV);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:01


In [36]:
query = """
CREATE INDEX IF NOT EXISTS idx_installments_payments_sk_id_prev
ON installments_payments(SK_ID_PREV);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:06


In [37]:
query = """
CREATE INDEX IF NOT EXISTS idx_bureau_balance_sk_id_bureau
ON bureau_balance(SK_ID_BUREAU);
"""
db.execute_query(query)

Запрос выполнен успешно
Время выполнения: 0:00:11
